In [17]:
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pypdf import PdfWriter
import io

In [18]:
df_tasks = pd.read_excel("../bert-tasks.xlsx")

In [19]:
df_tasks_label_mapping = pd.read_excel("../bert-tasks-class-label-mapping.xlsx")

In [20]:
df_tasks.head()

,id,dataset_name,huggingface_url,subset,classes,text_col,label_col,train_count,val_count,test_count,train_split,val_split,test_split,task_name,data_dir,is_integer_labels
0,1,cardiffnlp/tweet_eval,https://huggingface.co/datasets/cardiffnlp/twe...,emoji,20,text,label,45000,5000.0,50000.0,train,validation,test,cardiffnlp/tweet_eval@emoji,/Users/longpham28/Documents/GitHub/ntcir19-pre...,1
1,2,cardiffnlp/tweet_eval,https://huggingface.co/datasets/cardiffnlp/twe...,emotion,4,text,label,3260,374.0,1420.0,train,validation,test,cardiffnlp/tweet_eval@emotion,/Users/longpham28/Documents/GitHub/ntcir19-pre...,1
2,3,cardiffnlp/tweet_eval,https://huggingface.co/datasets/cardiffnlp/twe...,hate,2,text,label,9000,1000.0,2970.0,train,validation,test,cardiffnlp/tweet_eval@hate,/Users/longpham28/Documents/GitHub/ntcir19-pre...,1
3,4,cardiffnlp/tweet_eval,https://huggingface.co/datasets/cardiffnlp/twe...,irony,2,text,label,2860,955.0,784.0,train,validation,test,cardiffnlp/tweet_eval@irony,/Users/longpham28/Documents/GitHub/ntcir19-pre...,1
4,5,cardiffnlp/tweet_eval,https://huggingface.co/datasets/cardiffnlp/twe...,offensive,2,text,label,11900,1320.0,860.0,train,validation,test,cardiffnlp/tweet_eval@offensive,/Users/longpham28/Documents/GitHub/ntcir19-pre...,1


In [23]:
df_tasks_label_mapping.head(30)

,task_id,task_name,label,label_text
0,1,cardiffnlp/tweet_eval@emoji,0,❤
1,1,cardiffnlp/tweet_eval@emoji,1,😍
2,1,cardiffnlp/tweet_eval@emoji,2,😂
3,1,cardiffnlp/tweet_eval@emoji,3,💕
4,1,cardiffnlp/tweet_eval@emoji,4,🔥
5,1,cardiffnlp/tweet_eval@emoji,5,😊
6,1,cardiffnlp/tweet_eval@emoji,6,😎
7,1,cardiffnlp/tweet_eval@emoji,7,✨
8,1,cardiffnlp/tweet_eval@emoji,8,💙
9,1,cardiffnlp/tweet_eval@emoji,9,😘


In [25]:
tasks_label_mapping = {}
for _, row in df_tasks_label_mapping.iterrows():
    task_name = row["task_name"]
    df_tasks_label_mapping_task = df_tasks_label_mapping[df_tasks_label_mapping["task_name"] == task_name]
    label_mapping = dict(zip(df_tasks_label_mapping_task["label"], df_tasks_label_mapping_task["label_text"]))
    tasks_label_mapping[task_name] = label_mapping

In [26]:
tasks_label_mapping

{'cardiffnlp/tweet_eval@emoji': {0: '❤',
  1: '😍',
  2: '😂',
  3: '💕',
  4: '🔥',
  5: '😊',
  6: '😎',
  7: '✨',
  8: '💙',
  9: '😘',
  10: '📷',
  11: '🇸',
  12: '☀',
  13: '💜',
  14: '😉',
  15: '💯',
  16: '😁',
  17: '🎄',
  18: '📸',
  19: '😜'},
 'cardiffnlp/tweet_eval@emotion': {0: 'anger',
  1: 'joy',
  2: 'optimism',
  3: 'sadness'},
 'cardiffnlp/tweet_eval@hate': {0: 'non-hate', 1: 'hate'},
 'cardiffnlp/tweet_eval@irony': {0: 'non_irony', 1: 'irony'},
 'cardiffnlp/tweet_eval@offensive': {0: 'non-offensive', 1: 'offensive'},
 'cardiffnlp/tweet_eval@sentiment': {0: 'negative',
  1: 'neutral',
  2: 'positive'},
 'dair-ai/emotion@split': {0: 'adness',
  1: 'joy',
  2: 'love',
  3: 'anger',
  4: 'fear',
  5: 'surprise'},
 'google-research-datasets/poem_sentiment': {0: 'negative',
  1: 'positive',
  2: 'no impact',
  3: 'mixed'},
 'cornell-movie-review-data/rotten_tomatoes': {0: 'neg', 1: 'pos'},
 'clinc/clinc_oos@plus': {0: 'restaurant_reviews',
  1: 'nutrition_info',
  2: 'account_blocked'

In [27]:
tasks = df_tasks.to_dict(orient="records")

In [28]:
task = tasks[0]

In [29]:
def get_dataset(task):
    data_dir = Path(task["data_dir"])
    train_jsonl = data_dir / "train.jsonl"
    val_jsonl = data_dir / "val.jsonl"
    test_jsonl = data_dir / "test.jsonl"
    task_name = task["task_name"]

    train_df = pd.read_json(train_jsonl, lines=True)
    val_df = pd.read_json(val_jsonl, lines=True)
    test_df = pd.read_json(test_jsonl, lines=True)

    if not task_name in tasks_label_mapping:
        return train_df, val_df, test_df
    task_label_mapping = tasks_label_mapping[task_name]
    train_df["labels"] = train_df["labels"].map(task_label_mapping)
    val_df["labels"] = val_df["labels"].map(task_label_mapping)
    test_df["labels"] = test_df["labels"].map(task_label_mapping)
    return train_df, val_df, test_df

In [30]:
def create_plot(train_df, val_df, test_df, task_name="Task"):
    """
    Plots the label distribution for Train, Val, and Test splits in 3 rows.
    The X-axis order is locked to the Training set distribution.
    """

    # 1. Prepare Data
    # Storing in a dict allows us to loop through them easily
    splits = {
        "Train": train_df["labels"].value_counts(),
        "Val": val_df["labels"].value_counts(),
        "Test": test_df["labels"].value_counts(),
    }

    # Define colors for better visual separation
    colors = {"Train": "#636EFA", "Val": "#EF553B", "Test": "#00CC96"}

    # 2. Determine the Master Sort Order (based on Train)
    train_order = splits["Train"].index.tolist()

    # 3. Create Subplots
    fig = make_subplots(
        rows=3, cols=1, subplot_titles=[f"{name} Distribution" for name in splits.keys()], vertical_spacing=0.12
    )

    # 4. Add Traces (Loop instead of repeating code)
    for i, (name, counts) in enumerate(splits.items(), start=1):
        fig.add_trace(
            go.Bar(
                x=counts.index,
                y=counts.values,
                name=name,
                text=counts.values,
                textposition="outside",
                marker_color=colors.get(name),
            ),
            row=i,
            col=1,
        )

    # 5. Global Settings
    fig.update_layout(height=900, title_text=f"{task_name}", showlegend=False)  # Subplot titles are sufficient

    # Apply the Train sort order to ALL subplots
    fig.update_xaxes(categoryorder="array", categoryarray=train_order)

    # Ensure text labels don't get cut off
    fig.update_traces(cliponaxis=False)

    return fig

In [31]:
pdf_writer = PdfWriter()
for task in tasks:
    train_df, val_df, test_df = get_dataset(task)
    task_id = str(task["id"]).zfill(2)
    save_path = f"plots/task_{task_id}_distribution.png"

    task_name = f"Task {task_id}: {task['task_name']}"

    fig = create_plot(train_df, val_df, test_df, task_name=task_name)

    pdf_writer.append(io.BytesIO(fig.to_image(format="pdf")))

In [32]:
output_file_name = "label_distributions.pdf"
with open(output_file_name, "wb") as f:
    pdf_writer.write(f)